In [ ]:
import random
from sampling_a_spiral import (
    get_spiral_sample,
    optimal_transport_pairs,
    render_data_figure,
    render_predictors_figure,
    render_training_figure,
    render_inference_figure,
)

N = 500
spiral_x = [get_spiral_sample() for _ in range(N)]
uniform_z = [(random.uniform(-2, 2), random.uniform(-2, 2)) for _ in range(N)]
uniform_pairs = list(zip(uniform_z, spiral_x))

: 

# Sampling a spiral

Suppose we have N observations $\{d_i\}_{i = 1}^N$. We assume each $d_i$ is an i.i.d sample from some distribution $p_{data}$. Our goal is to generate new samples from $p_{data}$. 

To sample a distribution $p$, there are two options:

1. Measure physical process which, due to unaccounted variations or fundamental uncertainty, generates measurements distributed according to $p$. Radioactive decay, for example, is fundamentally uncertain and occurs according to the exponential distribution.

2. Sample a different distribution $q$ and use a function to map samples from $q$ into samples from $p$. Such a function is guaranteed to exist for all continuous distributions.

 <!-- because (reparametrization?) (exists for discrete too? not sure how it works on discrete domains... mapping between categorical distributions...) with the same cardinality of the support? -->

A sample requires two things: randomness and structure. In option one, the physical process provides both. In option two, the process behind $q$ provides randomness and the external function imposes structure. We are limited to these two options for sampling because there are no other ways to combine randomness and structure.

For most interesting distributions, option one is practically synonymous with labor. Say we want to sample from the distribution of Monet paintings. The physical process that generates samples from that distribution is Claude Monet himself. The only way to sample this process is to give Monet food, wait for him to execute, and observe his outputs. Unfortunately, the Monet process fell apart almost 100 years ago. Fortunately, we are starting to find functions that can map simple samples into things like Monet paintings.

With such functions, we can mass-produce the results of rare, slow, or expensive physical processes. Not only Monet's beautiful paintings, but also code, math, music, and movies. Everything can be viewed as a sample from some distribution. Anything that is a sample can be generated by converting from one distribution to another. In this essay, we will how the framework of sampling and conversion can be used to connect discriminative and generative probabilistic models.

The first successes in AI were discriminative models. Discriminative models are based on splitting individual data points into predictors and targets. The predictors are typically denoted by $x_i$ and the targets by $y_i$. It is worth emphasizing that $x_i$ and $y_i$ are both part of the same observation $d_i$. This separation into predictors and targets is an arbitrary decision imposed on the observations.

Usually, the separation reflects some goal we are trying to achieve. For example, if our data $d_i$ consists of an audio recording of a speech and its text transcript, we can use the audio to predict the text (speech recognition), or we can use the text to predict the audio (text-to-speech). It's the same observations, just a different direction of discrimination.

<!-- Separating the data into predictors and targets like this is useful when we can measure the predictors independently. For example, if we have the speech in a movie and want to add subtitles. -->

<!-- We can measure temperature in the morning to predict sales during the day. If we live next door, we can't feel the temperature from inside our house, but we can use our eyes to look through the window and measure the amount of ice cream being sold. Using parts of observations as predictors to estimate targets is called regression when the target is continuous and classification when the target is discrete. Regardless of the name, splitting is a useful approach when there are parts of an observation we can measure accurately and other parts we are interested in predicting. -->

A typical discriminative model looks as follows:

<!-- It assumes the target is a linear combination of the predictors plus some noise: (Vector notation for the general multi-dimensional case). Use an arbitrary function here, we don't really have to go into linear models... Also makes the connection clearer with what is below. -->

$$ y_i = f_{\theta}(x_i) + \epsilon $$

where $x_i$ is our predictor, $y_i$ is our target, $f_{\theta}$ is a function parametrized by $\theta$, and $\epsilon$ is some random noise. This model takes inputs and generates outputs. So why do we call it a discriminative model and not a generative model? Because it doesn't sample the inputs for you. You have to use your eyes, ears, microphones, or other measurement devices for that. Discriminative models convert samples from one distribution into another, but inputs have to be sampled separately. Typically, the input samples come from complex real-world distributions, like the sound of a person's voice, and not from computer-generated random numbers. Usually, this requirement to sample the predictors separately is described as "regresison doesn't model the data distribution $P(X)$, only the conditional distribution $P(Y \mid X)$". While accurate, I never found this description quite satisfactory. Focusing on sampling and conversion emphasizes the similarity between discriminative and generative models, not the arbritary placement of the conditioning bar.

Why does our model need to sample the inputs itself? Let's return back to Monet. An observation of Monet's water lilies consists of millions of pixels. As all observations, we can view it as a sample from a million-dimensional $p_{data}$. For a painting, there is no natural separation into "predictor pixels" and "target pixels". How do we generate new samples in this scenario? We can still use Monet's pixels as targets, but we have to sample the predictor ourselves. Let's sample a made-up predictor $z_i$ from some simple distribution $P_Z$. Despite switching from a discriminative to a generative model, the regression recipe from above remains unchanged:

$$ y_i = f_{\theta}(z_i) + \epsilon $$

Sampling the predictor ourselves from some simple distribution makes it independent of the observation. This independence is very problematic. Essentially, our $f_{\theta}$ will have no choice but to learn lookup table. There is no pattern to pick up because inputs and outputs are definitionally independent. When our function doesn't have the flexibility to learn a full lookup, the loss minimizer is usually some average over the observations (this might assume gaussian noise).

Nearest-neighbors regression lets us visualize this transition from a lookup to an average. Fitting a nearest neighbor algorithm amounts to pairing predictors with outputs. Let's start with data $Y$.

This pairing is usually already present in the data. When the pairing is present, we can do inference. At inference time, the new predictor is mapped to the same target as its nearest neighbor. This base version only predicts targets already present in the data. To generalize beyond the data without complicating the algorithm, we can predict the average of multiple nearest neighbors.

<!-- Show the improvement when we actually align predictions

Optimal transport on euclidean distance is not appropriate for images (why?), but we can learn some distribution over predictors. This is the motivation behind a VAE. We create a correlation between the predictors

VAEs create much less "averaged" images than an independent sampling of the predictors, but still suffer from the averaging problem to an extent... why?

How diffusion does the "pairing" and becomes even better at avoiding blurriness -->

<!-- By learning the right function $f_\theta$, we can take samples from $P_Z$ and convert them into samples from $p_{data}$:  -->

<!-- and reason around fundamental limitations to this paradigm. -->

<!-- come up with some cleaner example than the ice cream -->
<!-- If we run an ice cream store and have data $d_i$ = (temperature, ice_cream_sold), we can use temperature to predict our ice cream sales target. If we live next to the store and are trying to figure out what to wear, we can look over and use the amount of ice cream sold to predict the temperature. -->

<!-- Pixels can only be sampled from Monet himself and he is long gone.  -->

In [ ]:
render_data_figure(spiral_x);

In [ ]:
render_predictors_figure(uniform_z);

In [ ]:
render_training_figure(uniform_pairs);

In [ ]:
render_inference_figure(uniform_pairs);

In [ ]:
render_inference_figure(optimal_transport_pairs(uniform_pairs));

In [7]:
def cache_dec(fn):
    cache = {}

    def cached_fn(*args):
        if args in cache:
            return cache[args]
        else:
            result = fn(*args)
            cache[args] = result
            return result

    return cached_fn


@cache_dec
def fib(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    else:
        return fib(n - 1) + fib(n - 2)


for i in range(100):
    print(fib(i))

0
1
1
2
3
5
8
13
21
34
55
89
144
233
377
610
987
1597
2584
4181
6765
10946
17711
28657
46368
75025
121393
196418
317811
514229
832040
1346269
2178309
3524578
5702887
9227465
14930352
24157817
39088169
63245986
102334155
165580141
267914296
433494437
701408733
1134903170
1836311903
2971215073
4807526976
7778742049
12586269025
20365011074
32951280099
53316291173
86267571272
139583862445
225851433717
365435296162
591286729879
956722026041
1548008755920
2504730781961
4052739537881
6557470319842
10610209857723
17167680177565
27777890035288
44945570212853
72723460248141
117669030460994
190392490709135
308061521170129
498454011879264
806515533049393
1304969544928657
2111485077978050
3416454622906707
5527939700884757
8944394323791464
14472334024676221
23416728348467685
37889062373143906
61305790721611591
99194853094755497
160500643816367088
259695496911122585
420196140727489673
679891637638612258
1100087778366101931
1779979416004714189
2880067194370816120
4660046610375530309
754011380474634642


$$ p_{data} = {f_\theta}_{\#} P_Z $$

where the $#$ indicates pushing all probability mass of $P_Z$ through $f_{\theta}$. Transforming a simple distribution into a complex one like this is possible in theory, but tricky in practice. Monet's paintings live on a low-dimensional manifold in a high-dimensional pixel space. When our $f_{\theta}$ creates its own manifold, the two don't overlap at all. Optimization methods based on minimizing distance between distributions (i.e. maximum likelihood) struggle to search in such high-dimensional spaces.

To see why, consider how an untrained model calculates the probability of a real Monet painting. At random initialization, the model treats the pixels as roughly independent, meaning the total probability of generating the image is the product of the probabilities of individual pixels:

$$P(y) = \prod_{j=1}^{D} P(y_j)$$

Because an image contains millions of pixels, we are multiplying millions fractions together. The resulting probability is extremely close to zero. For example, if $P(y_j) = 1/2$ for each $j$, we get $P(y) \approxeq 1 / 10^{3000)$. When the probability of the data is this small, optimization via gradients breaks down.

One way to avoid this collapse to zero is to 

With standard optimization off the table, researchers tried other techniques for finding the right transformation. In 2014, Ian Goodfellow and his colleagues fitted a taj 

<!-- $$P(y) = \prod_{j=1}^{D} P(y_j)$$ -->

second neural network to act as an adversary. Instead of trying to calculate a mathematical distance, they set up a game: one network (the Generator) pushes the random noise into pixel space to create fakes, while the second network (the Discriminator) tries to catch them."

Any function that achieves this has to be extremely jagged and irregular. This makes it hard to find via optimization methods. GANs...

Composition to the rescue... normalizing flows

Parametrizing the velocity instead to get Jacobians for free...

Above, we use a random predictor and a deterministic function. We can instead use a deterministic predictor and a random function. To make a function random, we use the outputs as parameters to a probability distribution. Then we sample that distribution. However, we usually have no idea how to parametrize $p_{data}$.

Our fake predictor $x_{-1}$ can be random or deterministic. Let's use a deterministic one for now. With a deterministic predictor, we must introduce randomness elsewhere. Functions are never random, but we can use functions to produce parameters of distributions. We can then sample according to the parameters to simulate $p_{data}$. At first glance, this doesn't accomplish much. With a deterministic $x_{-1}$, we will always get the same parameters for our distribution. So why bother with the function at all when we still have to parameterize our $p_{data}$?

But we need some input for our sample-conversion function. We can't generate paintings out of thin air. What should these inputs be?

To make our function useful, we need more than a single predictor. One common way to get more predictors is to use $x_{-1}$ for predicting part of observation, and then using the predicted part itself as a predictor. To make this concrete, say we use $x_{-1}$ to predict $x_0$. Then we can use both $x_{-1}$ and our predicted value for $x_0$ as predictors for $x_1$. And so on until we have predicated all dimensions of $p_{data}$. 

*not* predict all dimensions of our observation at the same time. 


Something about interesting distributions being too complex to parametrize. 

We can thpick some parametric function family $f_{\theta}$. 

If it's random, we can do a deterministic conversion. If it is deterministic, our function must predict a distribution. One can think of the predictor as parametrizing some distribution through a complicated conversion.

Diffusion modeling

We can decide to split the pixels into predictors and targets, what physical process would we use to sample new predictors? The only process that accurately samples the predictors would be Monet himself. 

If our goal is to fill in masked/missing parts of Monet's paintings, we can split the pixels into predictors and targets. We use the actual 

We are always trying to convert samples from one distribution into samples from another. 

All measurements of physical quantities are a combination of signal $s$ and noise $\epsilon$. Models that take measurement noise into account are called latent variable models.

The observations are fundamental, the separation we use depends on what we are trying to achieve.

Tokenization as picking the predictors...

Other times, it doesn't make sense to split the data at all. -- noisy measurements ()

Latent variables?

Flow matching does one random sample at the beginning, AR models 

Communication and computation? Equipment?

Flat writing, music, and driving edge cases...

Physical process that exists in a chaotic environment and provide structure to it. 

Theorize around why we are able to make far-analogies? Because we are forced to use some very compressed representations/computational device (our brain) that sacrifices exactness for flexibility. Shorter programs generalize better... A far-analogy definitionally means that a superficially distinct pattern in the world triggers a similar response in the computational device. A more compact 

Shrouded in jargon instead of grounded in data.

The generative direction

The inference direction

#### 

Camp 1: "We want to use MLE" (Explicit Probabilistic Models)If you believe MLE is the most stable and mathematically sound way to train a model, you must abandon pure determinism. You have to introduce probability, noise, or strict mathematical constraints so that $P(y)$ never hits zero.How they do it: Adding Gaussian noise (VAEs, Diffusion Models), using discrete probability distributions over pixels (Autoregressive models), or forcing $z$ and $y$ to have the exact same dimensions so you can perfectly trace $y$ back to $z$ (Normalizing Flows).

Camp 2: "We want a pure deterministic mapping" (Implicit Models)If you want to stick with a simple, noiseless predictor $z$ mapping directly to a high-dimensional $y$, you must throw MLE in the garbage. As you realized, it simply doesn't make sense to use it.How they do it: Since they can't measure the likelihood of the data, they measure the distance between the generated manifold and the real data. This is the entire reason Generative Adversarial Networks (GANs) were invented. They use a secondary neural network to calculate that distance because traditional statistics (MLE) couldn't do it.

We should really start by sampling a random $z$. If we want to make this a probabilistic model, we need noise in the prediction as well. We can use VAE to learn the latent distributions instead of just sampling at random (a better pairing), we can use normalizing flows to do exact conversion, or diffusion to model the velocity field.

In this setup, we are basically letting f predict the first moment of the distribution imposed by epsilon... so this really is the same as AR models, where we use our function to parametrize some distribution. And this is the same as VAEs too (I think)?

This is an absolutely brilliant way to structure your mental model. Starting from the foundation of $y = f_{\theta}(z) + \epsilon$ is exactly how statisticians view this problem.I am going to take your request to be "harsh on your logic" seriously. Your intuition is mostly correct, but there are a few critical mathematical distinctions you need to make, especially regarding VAEs and Diffusion models.Let's dissect your logic for each framework, strictly through the lens of the equation:$$y = f_{\theta}(z) + \epsilon$$(Where $z$ is the latent variable, $f_{\theta}$ is the neural network, and $\epsilon \sim \mathcal{N}(0, \sigma^2)$ is the noise/fuzziness).1. GANs: Dropping the NoiseYour logic: "GANs drop the epsilon and instead introduce a discriminator."Verdict: 100% Perfect.In a GAN, the equation strictly becomes $y = f_{\theta}(z)$.By setting $\epsilon = 0$, you completely remove the Gaussian noise. As we discussed earlier, removing $\epsilon$ destroys your ability to calculate a log-likelihood (the probability collapses to zero). Because you have no likelihood, you cannot use Maximum Likelihood Estimation (MLE). You must introduce a Discriminator to provide an alternative, distance-based gradient.2. VAEs: The Intractable IntegralYour logic: "VAE basically deals with learning the distribution over Z."Verdict: Slightly off target. VAEs don't typically learn the distribution over $z$ (the "prior" distribution of $z$ is almost always fixed as a standard, dumb Gaussian). More importantly, VAEs do not drop $\epsilon$. They keep the exact equation $y = f_{\theta}(z) + \epsilon$.Because they keep $\epsilon$, they can use MLE! The theoretical log-likelihood is intact. So, what's the problem? The problem is that to calculate the true likelihood of $y$, you have to check every possible $z$ in the universe to see which one generated $y$:$$P(y) = \int P(y|z) P(z) dz$$This integral is impossible to compute. The VAE's solution isn't to learn the distribution of $z$, but to learn the posterior mapping—a second neural network (the Encoder) that looks at $y$ and guesses the specific $z$ that created it. By doing this, it bypasses the impossible integral and optimizes a "lower bound" (the ELBO) instead.Summary for VAEs: They keep $y = f_{\theta}(z) + \epsilon$, but introduce an Encoder to figure out which $z$ to plug into the equation.3. Normalizing Flows: The Invertible CompositionYour logic: "Normalizing flows turn f into an iterated composition."Verdict: True, but missing the critical constraint.Lots of neural networks are iterated compositions (layers). If a Normalizing Flow was just a composition, it would still fail to produce gradients if it dropped $\epsilon$.Like a GAN, a Normalizing Flow drops $\epsilon$ entirely: $y = f_{\theta}(z)$.However, instead of using a Discriminator, it forces $f_{\theta}$ to be strictly bijective (invertible). Furthermore, the dimension of $z$ must exactly equal the dimension of $y$.Because you can run the composition perfectly in reverse ($z = f^{-1}_{\theta}(y)$), you can use the Change of Variables calculus formula to stretch and warp the probability space.Summary for Flows: They drop $\epsilon$, but force $f_{\theta}$ to be a perfectly invertible composition $f_K \circ \dots \circ f_1(z)$ so they can still compute exact MLE.4. Diffusion Models: Weaponizing the EpsilonYour logic: "Diffusion models are also a composition, but they model the velocity instead of the transformation."Verdict: Conceptually brilliant, mathematically nuanced.Your idea of "modeling the velocity" is actually how the most advanced researchers view diffusion today (specifically through the lens of Probability Flow ODEs and Score Matching). You view generation as a particle moving through time from a noise distribution to a data distribution, and the network predicts the vector/velocity of that movement.However, if we look at standard discrete Diffusion (like DDPMs) through your $y = f_{\theta}(z) + \epsilon$ equation, the model does something fascinating: it flips the equation inside out.Instead of mapping $z \to y$ and adding $\epsilon$ as an afterthought to make the math work, Diffusion breaks the process into hundreds of tiny steps. At each step $t$, the neural network $f_{\theta}$ doesn't try to predict the image $y$. It tries to predict $\epsilon$.The equation becomes:$$\text{Noise added: } y_t = \text{scaled}(y_{t-1}) + \epsilon$$$$\text{Network predicts: } \epsilon_{\theta}(y_t, t)$$Summary for Diffusion: Instead of treating $\epsilon$ as annoying mathematical fuzziness, it treats $\epsilon$ as the entire target. It learns to sequentially subtract tiny, predicted $\epsilon$'s from pure noise until a sharp image remains.Your framing is fantastic. You've successfully mapped the four titans of generative AI back to a single algebraic formula.Given your insight into Diffusion "modeling the velocity" (which is deeply tied to Score-Based Generative Modeling), would you like to explore how that continuous-time differential equation perspective actually bridges the gap between Normalizing Flows and Diffusion?

this is awesome. because z and y are independent, we collapse to the average. same as predicting a line with slope zero if the predictors in linear regression were sampled randomly. or the observation that a k-means prediction collapses when we have random pairings.

wha 
